### Pasos para ejecutar este archivo

### instala python y demas dependencias 
- instalar python
- instalar python-venv
- instalar python-pip

### crear la carpeta del proyecto y abrirlo en vscode

### activar el entorno virtual
en Linux

 ```bash
python3 -m venv .venv
source .venv/bin/active
```

- crea el entorno virtual
- activalo

deberia aparecer algo como 

```bash
(.venv) usuario@pc:~/Documentos/AprendizajeComp$

```

### instala las dependencias para el notebook

```bash
# Actualizar pip dentro del entorno
python -m pip install --upgrade pip
# Instalar ipykernel (para que el venv pueda ser un kernel de Jupyter)
python -m pip install ipykernel

# registra el kernel con un nombre personalizable
python -m ipykernel install --user \\
  --name=aprendizajecomp \\
  --display-name="Python (AprendizajeComp)"


```

### Instalar dependencias
!pip install pandas numpy matplotlib scikit-learn joblib imbalanced-learn
Nota ejecutar solo una vez.

In [487]:
!pip install pandas numpy matplotlib scikit-learn joblib imbalanced-learn

In [488]:
import pandas as pd
import matplotlib as plt
from sklearn import tree
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

In [489]:
wine = pd.read_csv('winequality-red.csv')
wine.head(5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [490]:
wine.quality.value_counts()


quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

### identificacion de variables x, y 
como podemos ver en el dataset tenemos (1599,12), es decir 1599 filas y 12 columnas de las cuales 11 son datos que podemos considerar de entrada y el ultimo podemos considerar de salida.

ahora bien analizando el dataset podemos observar que tenemos varios niveles de calidad(observe la celda anterior), Ademas de que el dataset se encuentra desbalanceado.

esto lo que va a generar es que nuestro modelo no se entrene de la mejor manera, Cuando ejecutas wine.quality.value_counts(), ves que hay muchas más muestras de calidad 5-6 que de 3-4 o 8. El modelo tiende a aprender patrones de las clases mayoritarias (sobreajuste) e ignora las minorías. Resultado: predice mal los vinos buenos o malos. Por eso usamos SMOTE para generar sintéticamente más muestras de clases minoritarias, equilibrando el entrenamiento.

Ademas, "Simplificamos el problema de 6 clases a 2 clases binarias: vinos de baja calidad (≤5) vs alta calidad (>5). Esto reduce la complejidad del modelo y mejora el balance de clases."

In [491]:
# lo primero que vamos hacer es a simplificar las clases 
wine['quality_binary'] = (wine['quality'] >= 6).astype(int)
print(wine['quality_binary'].value_counts())


quality_binary
1    855
0    744
Name: count, dtype: int64


### explicacion de simplificacion

```python
wine['quality_binary'] = (wine['quality'] >= 6).astype(int)
                         │────────────────────────────────│
                         Parte que crea 0s y 1s
```

Parte 1 : ``` wine['quality'] >= 6 ```

```python
    # Si quality es:
3  → 3 >= 6?  → FALSE
4  → 4 >= 6?  → FALSE
5  → 5 >= 6?  → FALSE
6  → 6 >= 6?  → TRUE  ✓
7  → 7 >= 6?  → TRUE  ✓
8  → 8 >= 6?  → TRUE  ✓
9  → 9 >= 6?  → TRUE  ✓
```

``Resultado`` una lista de true y false
```
Original:  [3, 5, 6, 7, 8, 3, 6, ...]
Resultado: [False, False, True, True, True, False, True, ...]
```

Parte 2 ``` .astype(int) ```

```
True  → 1
False → 0
```

al final queda un arreglo de 0 y 1 0 = calidad baja 1 = calidad alta


In [492]:
wine.quality_binary.value_counts()

quality_binary
1    855
0    744
Name: count, dtype: int64

In [493]:
from sklearn.model_selection import train_test_split

In [494]:
# hacemos uso de random_state eso lo que significa es que siempre escoge el mismo conjunto de datos para las pruebas.
train, test = train_test_split(wine, test_size=0.30, random_state=42)
train.shape

(1119, 13)

In [495]:
test.shape


(480, 13)

In [496]:
x_train = train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y_train = train['quality_binary']

In [497]:
# Balanceo con SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(k_neighbors=2, random_state=42)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)

In [498]:
y_train_balanced.value_counts()

quality_binary
1    588
0    588
Name: count, dtype: int64

In [499]:
x_test = test[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y_test = test['quality_binary']

### ¿Por qué Gradient Boosting Classifier?

**Gradient Boosting** es un algoritmo de ensamble que construye múltiples árboles de decisión secuencialmente, donde cada árbol aprende de los errores del anterior. 

**Características clave para nuestro problema:**

1. **Captura relaciones complejas**: mi dataset tiene 11 características con relaciones no lineales que un árbol simple no puede capturar
2. **Robusto ante desbalances**: Aunque este usando SMOTE para balancear, Gradient Boosting maneja bien clases desbalanceadas
3. **Excelente en clasificación binaria**: es ideal para este algoritmo ya que solo estamos manejando 0=calidad baja 1=calidad alta
4. **Auto-corrección**: Cada nuevo árbol corrige errores anteriores, mejorando iterativamente la precisión

**Parámetros usados:**
- `n_estimators=100`: 100 árboles que se corrigen mutuamente
- `learning_rate=0.1`: Aprendizaje moderado para evitar sobreajuste
- `max_depth=5`: Árboles pequeños para generalizar mejor
- `random_state=42`: Asegura reproducibilidad. Con este valor fijo, el modelo siempre genera los mismos resultados, permitiendo comparar experimentos y validar que los cambios en el rendimiento se deben al algoritmo, no a la aleatoridad.

In [500]:
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5,random_state=42)
model.fit(x_train_balanced, y_train_balanced)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [501]:
y_pred = model.predict(x_test)
y_pred

array([0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0,

In [502]:
df_prediccion = pd.DataFrame(y_pred)
df_prediccion['real']=y_test.values
df_prediccion.head(10)

,0,real
0,0,1
1,0,0
2,0,1
3,0,0
4,1,1
5,0,0
6,0,0
7,0,0
8,1,0
9,1,1


In [503]:
model.score(x_test, y_test)*100

79.16666666666666

* el modelo tuvo un 61% de efectividad toca balancearlo o cambiar de modelo.